In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [37]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [10]:
columns = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
]
df = pd.read_parquet("drive/MyDrive/green_tripdata_2026-01.parquet", columns=columns)

In [11]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance
0,2026-01-01 00:27:58,2026-01-01 00:55:16,65,233,6.20
1,2026-01-01 00:44:33,2026-01-01 01:32:56,66,188,5.36
2,2026-01-01 00:23:45,2026-01-01 00:45:03,65,179,10.60
3,2026-01-01 00:44:33,2026-01-01 01:00:45,42,141,4.20
4,2026-01-01 00:46:04,2026-01-01 01:04:40,95,82,2.76


In [12]:
print(df.shape, df.size)

(40272, 5) 201360


In [13]:
df.isna().sum()

,0
lpep_pickup_datetime,0
lpep_dropoff_datetime,0
PULocationID,0
DOLocationID,0
trip_distance,0


In [16]:
pickup = pd.to_datetime(df["lpep_pickup_datetime"])
dropoff = pd.to_datetime(df["lpep_dropoff_datetime"])
df["duration"] = (dropoff - pickup).dt.total_seconds() / 60  # duarartion in minutes

df = df.rename(
    columns={
        "PULocationID": "PUlocationID",
        "DOLocationID": "DOlocationID",
    }
)
df["PU_DO"] = (
    df["PUlocationID"].astype("Int64").astype(str)
    + "_"
    + df["DOlocationID"].astype("Int64").astype(str)
)

# Remove malformed records before training.
df = df[
    df["duration"].between(1, 60)
    & df["trip_distance"].between(0.01, 100)
    & df["PUlocationID"].notna()
    & df["DOlocationID"].notna()
].copy()

features = ["PU_DO", "trip_distance"]
target = "duration"
print(f"Rows after filtering: {len(df):,}")
print(df[[*features, target]].describe(include="all"))

Rows after filtering: 37,533
        PU_DO  trip_distance      duration
count   37533   37533.000000  37533.000000
unique   5129            NaN           NaN
top     74_75            NaN           NaN
freq     1749            NaN           NaN
mean      NaN       2.988667     15.636257
std       NaN       3.010883     10.312261
min       NaN       0.010000      1.000000
25%       NaN       1.270000      8.466667
50%       NaN       2.000000     13.000000
75%       NaN       3.410000     19.666667
max       NaN      37.390000     60.000000


In [20]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PUlocationID,DOlocationID,trip_distance,duration,PU_DO
0,2026-01-01 00:27:58,2026-01-01 00:55:16,65,233,6.20,27.300000,65_233
1,2026-01-01 00:44:33,2026-01-01 01:32:56,66,188,5.36,48.383333,66_188
2,2026-01-01 00:23:45,2026-01-01 00:45:03,65,179,10.60,21.300000,65_179
3,2026-01-01 00:44:33,2026-01-01 01:00:45,42,141,4.20,16.200000,42_141
4,2026-01-01 00:46:04,2026-01-01 01:04:40,95,82,2.76,18.600000,95_82


In [28]:
df = df.drop(columns=["PUlocationID", "DOlocationID"])

In [29]:
df.head()

,trip_distance,duration,PU_DO
0,6.20,27.300000,65_233
1,5.36,48.383333,66_188
2,10.60,21.300000,65_179
3,4.20,16.200000,42_141
4,2.76,18.600000,95_82


## **Data Spliting**

In [31]:
X = df.drop(columns="duration")
y = df["duration"]

In [33]:
y

,duration
0,27.300000
1,48.383333
2,21.300000
3,16.200000
4,18.600000
...,...
40267,15.000000
40268,10.000000
40269,25.983333
40270,25.000000


In [34]:
# Convert X to dictionaries
X_dicts = X.to_dict(orient="records")

# DictVectorizer
dv = DictVectorizer()

X_vec = dv.fit_transform(X_dicts)

# Chronological split
split_index = int(X_vec.shape[0] * 0.8)

X_train = X_vec[:split_index]
X_test = X_vec[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

## **Base Model**

In [38]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)

Root Mean Squared Error (RMSE): 10.029844217359013
Mean Absolute Error (MAE): 6.860088089016162
